In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
from math import sqrt

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [3]:
def combine_x_plus_y_divided_by_2(x, y, x_err, y_err):
    central_value = (x+y)/2
    error = sqrt(x_err**2 +  y_err**2)/2
    return central_value, error

In [4]:
def correct_Acp_stats( Araw, Araw_err, Aref, Aref_err, Aref_pdg, Aref_K_mix):
    final_Acp = Araw - Aref + Aref_pdg + Aref_K_mix
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [5]:
def correct_Acp_stats_no_Kmix( Araw, Araw_err, Aref, Aref_err, Aref_pdg):
    final_Acp = Araw - Aref + Aref_pdg 
    final_Acp_err = sqrt(Araw_err**2 + Aref_err**2)

    return final_Acp, final_Acp_err

In [6]:
def delta_Acp_sys_unc(A_original, A_original_error, A, A_error):
    delta_Acp = A - A_original
    if A_original_error > A_error:
        delta_Acp_error = sqrt(A_original_error**2 - A_error**2)
    elif A_original_error < A_error:
        delta_Acp_error = sqrt(A_error**2 - A_original_error**2)
    else: 
        print("Error: unable to proceed.")
        sys.exit()

    # print(f"delta_Acp: {delta_Acp}, delta_Acp_error: {delta_Acp_error}")
    print(f"Original Acp: {A_original * 100:.5f}%, Original Acp error: {A_original_error * 100:.5f}%")
    print(f"Acp: {A * 100:.5f}%, Acp error: {A_error * 100:.5f}%")
    print(f"delta_Acp: {delta_Acp * 100:.5f}%, delta_Acp_error: {delta_Acp_error * 100:.5f}%")
    return delta_Acp, delta_Acp_error

# MC15 full

## Acp(D+ -> eta K+)

### eta -> gg

In [7]:
#fitv15
Araw_gg_cms_plus = 0.1033505940299766
Araw_gg_cms_plus_error = 0.06527247319819734
Araw_gg_cms_minus = 0.10118610443834708
Araw_gg_cms_minus_error = 0.06259799759511811
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 10.22683%, Araw_gg_stats_error: 4.52189%


In [8]:
#fitv3
Aref_gg_cms_plus = -0.000891053282276677
Aref_gg_cms_plus_error = 0.002591316856349276
Aref_gg_cms_minus = 0.01696678351479708 
Aref_gg_cms_minus_error = 0.002695345787618729
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.80379%, Aref_gg_stats_error: 0.18695%


In [10]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)

print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value}, Acp_etapip_gg_error: {Acp_etapip_gg_error}")
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: 0.09423048411790164, Acp_etapip_gg_error: 0.045257554265262276
Acp_etapip_gg_value: 9.42305%, Acp_etapip_gg_error: 4.52576%


### eta -> pipipi

In [14]:
#fitv15
Araw_3pi_cms_plus = -0.030924128438692522
Araw_3pi_cms_plus_error = 0.06460802044084245
Araw_3pi_cms_minus = 0.054827044453552576
Araw_3pi_cms_minus_error = 0.06262481621735487
Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 1.19515%, Araw_3pi_stats_error: 4.49891%


In [15]:
#fitv3
Aref_3pi_cms_plus = 0.00013218589725561003
Aref_3pi_cms_plus_error =   0.002166365637515913
Aref_3pi_cms_minus = 0.01753684985818671
Aref_3pi_cms_minus_error =   0.0022719620271573465
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.88345%, Aref_pipipi_stats_error: 0.15696%


In [17]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value}, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error}")
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 0.0031169401297088672, Acp_etapip_pipipi_error: 0.045016438283892596
Acp_etapip_pipipi_value: 0.31169%, Acp_etapip_pipipi_error: 4.50164%


In [25]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: 4.84304%, Acp_combined_error: 3.19163%


## Acp(Ds+ -> eta K+)

### eta -> gg

In [11]:
#fitv15
Araw_gg_cms_plus = 0.03139616818379132
Araw_gg_cms_plus_error = 0.015114988331443901
Araw_gg_cms_minus = 0.043925778660172776
Araw_gg_cms_minus_error = 0.015532364950835393
Araw_gg , Araw_gg_stats_error= combine_x_plus_y_divided_by_2(Araw_gg_cms_plus,Araw_gg_cms_minus, Araw_gg_cms_plus_error,Araw_gg_cms_minus_error )

print(f"Araw_gg: {Araw_gg * 100:.5f}%, Araw_gg_stats_error: {Araw_gg_stats_error * 100:.5f}%")

Araw_gg: 3.76610%, Araw_gg_stats_error: 1.08365%


In [12]:
#fitv3
Aref_gg_cms_plus = -0.000891053282276677
Aref_gg_cms_plus_error = 0.002591316856349276
Aref_gg_cms_minus = 0.01696678351479708 
Aref_gg_cms_minus_error = 0.002695345787618729
Aref_gg, Aref_gg_stats_error = combine_x_plus_y_divided_by_2(Aref_gg_cms_plus,Aref_gg_cms_minus, Aref_gg_cms_plus_error,Aref_gg_cms_minus_error )

print(f"Aref_gg: {Aref_gg * 100:.5f}%, Aref_gg_stats_error: {Aref_gg_stats_error * 100:.5f}%")

Aref_gg: 0.80379%, Aref_gg_stats_error: 0.18695%


In [13]:
Aref_gg_pdg = 0
Acp_etapip_gg_value, Acp_etapip_gg_error = correct_Acp_stats_no_Kmix(Araw_gg, Araw_gg_stats_error, Aref_gg, Aref_gg_stats_error, Aref_gg_pdg)
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value}, Acp_etapip_gg_error: {Acp_etapip_gg_error}")
print(f"Acp_etapip_gg_value: {Acp_etapip_gg_value * 100:.5f}%, Acp_etapip_gg_error: {Acp_etapip_gg_error * 100:.5f}%")

Acp_etapip_gg_value: 0.029623108305721846, Acp_etapip_gg_error: 0.01099655679281541
Acp_etapip_gg_value: 2.96231%, Acp_etapip_gg_error: 1.09966%


### eta -> pipipi

In [18]:
#fitv15
Araw_3pi_cms_plus = -0.002371357302004662
Araw_3pi_cms_plus_error = 0.01815498437744717
Araw_3pi_cms_minus = 0.036714900390675265
Araw_3pi_cms_minus_error = 0.018595116979859987
Araw_3pi , Araw_3pi_stats_error = combine_x_plus_y_divided_by_2(Araw_3pi_cms_plus,Araw_3pi_cms_minus, Araw_3pi_cms_plus_error,Araw_3pi_cms_minus_error )

print(f"Araw_3pi: {Araw_3pi * 100:.5f}%, Araw_3pi_stats_error: {Araw_3pi_stats_error * 100:.5f}%")

Araw_3pi: 1.71718%, Araw_3pi_stats_error: 1.29941%


In [19]:
#fitv3
Aref_3pi_cms_plus = 0.00013218589725561003
Aref_3pi_cms_plus_error =   0.002166365637515913
Aref_3pi_cms_minus = 0.01753684985818671
Aref_3pi_cms_minus_error =   0.0022719620271573465  
Aref_pipipi, Aref_pipipi_stats_error = combine_x_plus_y_divided_by_2(Aref_3pi_cms_plus,Aref_3pi_cms_minus, Aref_3pi_cms_plus_error,Aref_3pi_cms_minus_error )

print(f"Aref_pipipi: {Aref_pipipi * 100:.5f}%, Aref_pipipi_stats_error: {Aref_pipipi_stats_error * 100:.5f}%")

Aref_pipipi: 0.88345%, Aref_pipipi_stats_error: 0.15696%


In [20]:
Aref_pipipi_pdg = 0
Acp_etapip_pipipi_value, Acp_etapip_pipipi_error =correct_Acp_stats_no_Kmix(Araw_3pi, Araw_3pi_stats_error, Aref_pipipi, Aref_pipipi_stats_error, Aref_pipipi_pdg)

print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value}, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error}")
print(f"Acp_etapip_pipipi_value: {Acp_etapip_pipipi_value * 100:.5f}%, Acp_etapip_pipipi_error: {Acp_etapip_pipipi_error * 100:.5f}%")

Acp_etapip_pipipi_value: 0.008337253666614142, Acp_etapip_pipipi_error: 0.013088513903116376
Acp_etapip_pipipi_value: 0.83373%, Acp_etapip_pipipi_error: 1.30885%


In [32]:
combined_central_value, combined_error = combine_error_weighted(Acp_etapip_gg_value, Acp_etapip_pipipi_value, Acp_etapip_gg_error, Acp_etapip_pipipi_error)

print(f"Acp_combined: {combined_central_value * 100:.5f}%, Acp_combined_error: {combined_error * 100:.5f}%")

Acp_combined: 2.08152%, Acp_combined_error: 0.84194%
